# 04 — CatBoost : Feature Selection Validation + Optuna

**Plan** :
1. Baseline CatBoost with les 377 features (params par défaut)
2. Test with SHAP top-150 / top-100 / top-50 → trouver le sweet spot
3. Optuna (80 trials) on le meilleur feature set
4. Analyse : feature importance CatBoost vs SHAP, residuals, RMSE par heure

In [1]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import time
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_data, merge_train
from src.feature_engineering import build_features
from src.validation import create_holdout_split, split_X_y, evaluate
from src.models.catboost_model import CatBoostForecaster

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

In [2]:
# Load data
x_train, y_train, x_test = load_data(data_dir='../data/raw')
train = merge_train(x_train, y_train)
df = build_features(train, config)
train_df, val_df = create_holdout_split(df, config)

# Split X/y for both targets
X_tr_fr, y_tr_fr = split_X_y(train_df, 'fr_spot')
X_val_fr, y_val_fr = split_X_y(val_df, 'fr_spot')
X_tr_uk, y_tr_uk = split_X_y(train_df, 'uk_spot')
X_val_uk, y_val_uk = split_X_y(val_df, 'uk_spot')

print(f"Features: {X_tr_fr.shape[1]}")
print(f"Train: {len(X_tr_fr):,} | Val: {len(X_val_fr):,}")

Holdout split @ 2024-02-01
  Train: 13,921 rows  (2022-07-01 → 2024-01-31)
  Val:   3,623 rows  (2024-02-01 → 2024-06-30)
Features: 377
Train: 13,921 | Val: 3,623


---
## 1. SHAP Ranking (for les paliers de features)

On recalcule le ranking SHAP with un LightGBM rapide, then on teste CatBoost with different sous-ensembles.

In [3]:
import lightgbm as lgb
import shap

def get_shap_ranking(X_tr, y_tr, X_val, y_val, label=''):
    """Quick LightGBM + SHAP → feature ranking."""
    model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=20, verbose=-1, n_jobs=-1
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_val)
    imp = pd.Series(np.abs(sv).mean(axis=0), index=X_tr.columns).sort_values(ascending=False)
    
    rmse = np.sqrt(np.mean((y_val - model.predict(X_val))**2))
    print(f"[{label}] LightGBM RMSE={rmse:.2f}, best_iter={model.best_iteration_}")
    return imp

print("Computing SHAP rankings...")
shap_rank_fr = get_shap_ranking(X_tr_fr, y_tr_fr, X_val_fr, y_val_fr, 'FR')
shap_rank_uk = get_shap_ranking(X_tr_uk, y_tr_uk, X_val_uk, y_val_uk, 'UK')

Computing SHAP rankings...
[FR] LightGBM RMSE=30.42, best_iter=344
[UK] LightGBM RMSE=12.28, best_iter=498


---
## 2. CatBoost — Test par paliers de features

On entraîne CatBoost with les **mêmes hyperparamètres par défaut** but en variant le nombre de features.
Objectif : trouver le sweet spot between trop de bruit (377) et trop peu de signal (50).

In [4]:
# Test different feature set sizes — fine-grained sweep
feature_counts = [377, 300, 250, 200, 175, 150, 125, 100, 85, 75, 60, 50, 40, 30, 20, 10]
results = []

for target, X_tr, y_tr, X_val, y_val, shap_rank in [
    ('fr_spot', X_tr_fr, y_tr_fr, X_val_fr, y_val_fr, shap_rank_fr),
    ('uk_spot', X_tr_uk, y_tr_uk, X_val_uk, y_val_uk, shap_rank_uk),
]:
    print(f"\n{'='*60}")
    print(f"  {target.upper()} — Feature Set Comparison ({len(feature_counts)} paliers)")
    print(f"{'='*60}")
    
    for n_feat in feature_counts:
        if n_feat >= len(shap_rank):
            cols = X_tr.columns.tolist()
            actual_n = len(cols)
        else:
            cols = shap_rank.head(n_feat).index.tolist()
            actual_n = n_feat
        
        t0 = time.time()
        model = CatBoostForecaster(config, target=target)
        model.fit(X_tr[cols], y_tr, X_val[cols], y_val)
        
        preds = model.predict(X_val[cols])
        metrics = evaluate(y_val, preds, tag=f"{target}_{actual_n}feat")
        elapsed = time.time() - t0
        
        best_iter = model.model.get_best_iteration()
        results.append({
            'target': target,
            'n_features': actual_n,
            'rmse': metrics['rmse'],
            'mae': metrics['mae'],
            'smape': metrics['smape'],
            'best_iter': best_iter,
            'time_s': elapsed,
        })
        print(f"  → {actual_n:>3d} features: RMSE={metrics['rmse']:.3f} | "
              f"MAE={metrics['mae']:.3f} | iter={best_iter} | {elapsed:.0f}s")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print(results_df.to_string(index=False))


  FR_SPOT — Feature Set Comparison (16 paliers)
0:	learn: 143.4807974	test: 127.9930559	best: 127.9930559 (0)	total: 123ms	remaining: 10m 14s
100:	learn: 25.9928982	test: 41.7964355	best: 41.7964355 (100)	total: 2.49s	remaining: 2m
200:	learn: 19.2830532	test: 34.6496255	best: 34.6496255 (200)	total: 4.81s	remaining: 1m 54s
300:	learn: 16.8682859	test: 33.3806871	best: 33.3806871 (300)	total: 7.16s	remaining: 1m 51s
400:	learn: 15.0614005	test: 32.9866132	best: 32.9812730 (392)	total: 9.48s	remaining: 1m 48s
500:	learn: 13.6110512	test: 32.4538283	best: 32.4374337 (495)	total: 11.8s	remaining: 1m 45s
600:	learn: 12.5048339	test: 32.2403929	best: 32.2403929 (600)	total: 14.1s	remaining: 1m 42s
700:	learn: 11.5679697	test: 32.0374707	best: 32.0374707 (700)	total: 16.4s	remaining: 1m 40s
800:	learn: 10.7992587	test: 31.9028604	best: 31.9015990 (798)	total: 18.7s	remaining: 1m 37s
900:	learn: 10.1323722	test: 31.8395196	best: 31.8322318 (868)	total: 21s	remaining: 1m 35s
1000:	learn: 9.56

KeyboardInterrupt: 

In [ ]:
# Visualize RMSE vs number of features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, target, label in [(axes[0], 'fr_spot', 'FR'), (axes[1], 'uk_spot', 'UK')]:
    sub = results_df[results_df['target'] == target]
    
    ax.plot(sub['n_features'], sub['rmse'], 'bo-', linewidth=2, markersize=8)
    
    # Mark the best
    best_idx = sub['rmse'].idxmin()
    best = sub.loc[best_idx]
    ax.scatter([best['n_features']], [best['rmse']], color='red', s=150, zorder=5, marker='*')
    ax.annotate(f"Best: {best['rmse']:.3f}\n({int(best['n_features'])} feat)",
                xy=(best['n_features'], best['rmse']),
                xytext=(20, 20), textcoords='offset points',
                fontsize=11, color='red',
                arrowprops=dict(arrowstyle='->', color='red'))
    
    ax.set_xlabel('Number of features')
    ax.set_ylabel('Validation RMSE (EUR)')
    ax.set_title(f'CatBoost RMSE vs Feature Count — {label}', fontsize=13)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/catboost_feature_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
for target in ['fr_spot', 'uk_spot']:
    sub = results_df[results_df['target'] == target]
    best = sub.loc[sub['rmse'].idxmin()]
    worst = sub.loc[sub['rmse'].idxmax()]
    print(f"\n{target}: best={best['rmse']:.3f} ({int(best['n_features'])} feat) | "
          f"worst={worst['rmse']:.3f} ({int(worst['n_features'])} feat) | "
          f"Δ={worst['rmse']-best['rmse']:.3f}")

---
## 3. Optuna — Optimisation on le meilleur feature set

On prend le nombre de features qui a donné le meilleur RMSE et on lance Optuna (80 trials) for optimiser les hyperparamètres.

In [ ]:
# Select best feature count per target
best_n_fr = int(results_df[results_df['target'] == 'fr_spot'].loc[
    results_df[results_df['target'] == 'fr_spot']['rmse'].idxmin(), 'n_features'])
best_n_uk = int(results_df[results_df['target'] == 'uk_spot'].loc[
    results_df[results_df['target'] == 'uk_spot']['rmse'].idxmin(), 'n_features'])

print(f"Best feature count: FR={best_n_fr}, UK={best_n_uk}")

# Select features
if best_n_fr >= len(shap_rank_fr):
    cols_fr = X_tr_fr.columns.tolist()
else:
    cols_fr = shap_rank_fr.head(best_n_fr).index.tolist()

if best_n_uk >= len(shap_rank_uk):
    cols_uk = X_tr_uk.columns.tolist()
else:
    cols_uk = shap_rank_uk.head(best_n_uk).index.tolist()

print(f"FR features: {len(cols_fr)} | UK features: {len(cols_uk)}")

In [ ]:
# Optuna optimization — FR
print("="*60)
print("  OPTUNA — FR")
print("="*60)

model_fr = CatBoostForecaster(config, target='fr_spot')
t0 = time.time()
best_params_fr = model_fr.optimize(
    X_tr_fr[cols_fr], y_tr_fr,
    X_val_fr[cols_fr], y_val_fr,
    n_trials=80
)
print(f"Optuna FR done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Optuna optimization — UK
print("="*60)
print("  OPTUNA — UK")
print("="*60)

model_uk = CatBoostForecaster(config, target='uk_spot')
t0 = time.time()
best_params_uk = model_uk.optimize(
    X_tr_uk[cols_uk], y_tr_uk,
    X_val_uk[cols_uk], y_val_uk,
    n_trials=80
)
print(f"Optuna UK done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Final fit with best params
print("\n" + "="*60)
print("  FINAL FIT WITH OPTUNA PARAMS")
print("="*60)

model_fr.fit(X_tr_fr[cols_fr], y_tr_fr, X_val_fr[cols_fr], y_val_fr)
preds_fr = model_fr.predict(X_val_fr[cols_fr])
metrics_fr = evaluate(y_val_fr, preds_fr, tag='CatBoost_FR_optuna')

model_uk.fit(X_tr_uk[cols_uk], y_tr_uk, X_val_uk[cols_uk], y_val_uk)
preds_uk = model_uk.predict(X_val_uk[cols_uk])
metrics_uk = evaluate(y_val_uk, preds_uk, tag='CatBoost_UK_optuna')

combined = (metrics_fr['rmse'] + metrics_uk['rmse']) / 2
print(f"\n  Combined RMSE = ({metrics_fr['rmse']:.3f} + {metrics_uk['rmse']:.3f}) / 2 = {combined:.3f}")

In [ ]:
# Save models
model_fr.save('../outputs/models/catboost_fr.cbm')
model_uk.save('../outputs/models/catboost_uk.cbm')

# Save feature lists
import json
feature_config = {
    'fr_spot': {'features': cols_fr, 'n_features': len(cols_fr), 'best_params': best_params_fr},
    'uk_spot': {'features': cols_uk, 'n_features': len(cols_uk), 'best_params': best_params_uk},
}
with open('../outputs/models/catboost_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print("Models and config saved.")

---
## 4. Analyse des résultats

In [ ]:
# Feature importance: CatBoost vs SHAP
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

for ax, model, shap_rank, label in [
    (axes[0], model_fr, shap_rank_fr, 'FR'),
    (axes[1], model_uk, shap_rank_uk, 'UK'),
]:
    cb_imp = model.feature_importances_.head(25)
    cb_imp_norm = cb_imp / cb_imp.sum()
    
    cb_imp_norm.iloc[::-1].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'CatBoost Feature Importance — {label} (top 25)', fontsize=13)
    ax.set_xlabel('Relative importance')

plt.tight_layout()
plt.savefig('../outputs/catboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare rankings
for model, shap_rank, label in [
    (model_fr, shap_rank_fr, 'FR'),
    (model_uk, shap_rank_uk, 'UK'),
]:
    cb_top20 = set(model.feature_importances_.head(20).index)
    shap_top20 = set(shap_rank.head(20).index)
    overlap = cb_top20 & shap_top20
    print(f"\n{label}: CatBoost top-20 ∩ SHAP top-20 = {len(overlap)}/20")
    print(f"  Common: {sorted(overlap)}")
    print(f"  CatBoost only: {sorted(cb_top20 - shap_top20)}")
    print(f"  SHAP only:     {sorted(shap_top20 - cb_top20)}")

In [ ]:
# RMSE by hour — where does the model struggle?
hours_val = pd.to_datetime(val_df['datetime_CET']).dt.hour.values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_true, preds, label in [
    (axes[0], y_val_fr, preds_fr, 'FR'),
    (axes[1], y_val_uk, preds_uk, 'UK'),
]:
    hourly = {}
    for h in range(24):
        mask = hours_val == h
        if mask.sum() > 0:
            hourly[h] = np.sqrt(np.mean((y_true.values[mask] - preds[mask])**2))
    
    hours = list(hourly.keys())
    rmses = list(hourly.values())
    
    colors = ['#e74c3c' if r > np.mean(rmses) * 1.2 else '#3498db' for r in rmses]
    ax.bar(hours, rmses, color=colors)
    ax.axhline(np.mean(rmses), color='black', linestyle='--', alpha=0.5, label=f'Mean={np.mean(rmses):.1f}')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('RMSE (EUR)')
    ax.set_title(f'RMSE by Hour — {label}', fontsize=13)
    ax.legend()
    ax.set_xticks(range(24))

plt.tight_layout()
plt.savefig('../outputs/catboost_rmse_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print("Rouge = heures problématiques (RMSE > 120% de la moyenne)")

In [ ]:
# Residual analysis — scatter + distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for col, (y_true, preds, label) in enumerate([
    (y_val_fr, preds_fr, 'FR'),
    (y_val_uk, preds_uk, 'UK'),
]):
    residuals = y_true.values - preds
    
    # Scatter: predicted vs actual
    ax = axes[0, col]
    ax.scatter(y_true, preds, alpha=0.15, s=5, c='steelblue')
    lims = [min(y_true.min(), preds.min()), max(y_true.max(), preds.max())]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual (EUR)')
    ax.set_ylabel('Predicted (EUR)')
    ax.set_title(f'Predicted vs Actual — {label}', fontsize=13)
    
    # Residual distribution
    ax = axes[1, col]
    ax.hist(residuals, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', linewidth=1)
    ax.axvline(np.mean(residuals), color='orange', linewidth=1, linestyle='--',
               label=f'Mean={np.mean(residuals):.2f}')
    ax.set_xlabel('Residual (EUR)')
    ax.set_ylabel('Count')
    ax.set_title(f'Residual Distribution — {label}', fontsize=13)
    ax.legend()

plt.tight_layout()
plt.savefig('../outputs/catboost_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

# Bias check
for y_true, preds, label in [(y_val_fr, preds_fr, 'FR'), (y_val_uk, preds_uk, 'UK')]:
    res = y_true.values - preds
    print(f"{label}: mean_bias={np.mean(res):.2f} | median_bias={np.median(res):.2f} | "
          f"std={np.std(res):.2f} | skew={pd.Series(res).skew():.2f}")

In [ ]:
# Time series plot — predictions vs actual over val period
val_dates = pd.to_datetime(val_df['datetime_CET'])

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

for ax, y_true, preds, label in [
    (axes[0], y_val_fr, preds_fr, 'FR'),
    (axes[1], y_val_uk, preds_uk, 'UK'),
]:
    # Plot 2 weeks for readability
    n_points = 24 * 14  # 2 weeks
    ax.plot(val_dates.values[:n_points], y_true.values[:n_points], 
            'b-', alpha=0.7, linewidth=1, label='Actual')
    ax.plot(val_dates.values[:n_points], preds[:n_points], 
            'r-', alpha=0.7, linewidth=1, label='CatBoost')
    ax.set_ylabel('Spot Price (EUR)')
    ax.set_title(f'{label} — First 2 weeks of validation', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/catboost_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Résumé

### Results clés :
- **Feature sweep** : quel nombre de features est optimal for FR et UK
- **Optuna** : hyperparamètres optimisés for chaque cible
- **CatBoost vs SHAP** : les feature importances convergent-elles ?
- **RMSE par heure** : quelles heures sont les plus difficiles (peak hours ?)
- **Résidus** : biais systématique ou distribution propre ?

### Prochaines étapes :
1. LightGBM with loss MAE (diversité d'erreurs)
2. Ensemble Ridge (CatBoost + LightGBM)
3. Post-processing (clipping + hourly bias correction)